# 04 · ephys baseline: spike detection, silent / tonic, baseline features

This notebook analyses the **spontaneous baseline recordings** (A1 / M2, no current injected). For each recording
it uses the **last 100 s**, detects action potentials, classifies the recording as **silent** or **tonic**, and
measures **Vm** (spikes removed), **firing rate**, **ISI CV** and the **AP waveform**.

Input: `02_ephys_anomaly-fix/output/analysis_ready.csv` (only recordings whose problems are resolved).
Settings: `config.yaml` in this folder. Method: `README.md`.

| Step | What happens | Review |
|---|---|---|
| 1 | select the ready baseline recordings | which files |
| 2 | **detect spikes in every file and save a detection figure (PNG + SVG)** | open the figures: is every spike found, is each threshold right? |
| 3 | feature table, from the same spikes | class, rate, Vm, flags |
| 4 | save | |

## Setup
`OVERWRITE = True` replaces existing output files. Close them in Excel first.

In [ ]:
OVERWRITE = True

from pathlib import Path

import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

from ephys_analysis.baseline import BaselineSettings, detect_all, feature_table
from ephys_analysis.spikes import SpikeSettings
from ephys_analysis.utils import RunLog, check_writable, find_repo_root, load_config, repo_path, stamp, write_csv

REPO = find_repo_root()
cfg = load_config(REPO / "04_ephys_baseline")
sp = SpikeSettings.from_config(cfg["spikes"])
bs = BaselineSettings.from_config(cfg["baseline"])
out = REPO / "04_ephys_baseline" / "output"
figures = out / "figures"
targets = [out / "spikes.csv", out / "detection.csv", out / "baseline_features.csv"]
check_writable(targets, OVERWRITE)
log = RunLog(cfg, out)
print(sp)
print(bs)

## Step 1 · Select the baseline recordings
Only recordings with `family = spontaneous` that 02 marked ready. The voltage channel and any correction
(gain, bias current) come from `analysis_ready.csv`.

In [ ]:
ready = pd.read_csv(repo_path(cfg, cfg["recordings_table"]), dtype={"cell_tag": "string"})
selection = ready[ready["family"].isin(cfg["families"])].reset_index(drop=True)
display(selection[["recording_id", "cell_tag", "abf_filename", "voltage_source", "voltage_scale", "bias_current_pA", "corrections"]])

## Step 2 · Detect spikes and save a detection figure for every file
**Spike detection** (adapted from the Allen Institute IPFX extractor):
1. a spike starts where dV/dt rises above **20 mV/ms**; its peak must be above −20 mV and ≥ 20 mV above threshold;
2. **threshold** = the point just before the spike where dV/dt is **5 % of that spike's maximum upstroke**;
3. **half-width** = width at half-way between threshold and peak (interpolated);
4. **AHP** = lowest voltage within 50 ms after the peak, relative to threshold.

Spikes with a slow upstroke (< 50 mV/ms) or a half-width > 5 ms are counted but **flagged** (grey in the figure),
and their waveform is not used.

One figure per recording is saved to `output/figures/<recording_id>.png` and `.svg`:
the whole recording (window shaded, spikes marked), every spike aligned to its threshold, and the phase plot.

In [ ]:
spikes, detection = detect_all(selection, sp, bs, log, figures, tuple(cfg["figure_formats"]))
display(detection)

In [ ]:
# Review: every detection figure. Check that each spike is marked and each threshold (black dot) sits at the
# foot of the upstroke. If detection is wrong, change `spikes:` in config.yaml and re-run Step 2.
for f in detection["figure"].dropna():
    if f:
        print(Path(f).name)
        display(Image(filename=f, width=1000))

**Checkpoint.** Only continue when the detection is right: the feature table below is computed from
exactly these spikes.

## Step 3 · Feature table
**Silent** = no more than `silent_max_spikes` (10) spikes in the 100 s window; otherwise **tonic**.
**Vm** = median of the window after removing each spike (5 ms before to 50 ms after threshold). In a firing
cell this is not a true resting potential, so it is called `vm_baseline_mV`. The junction potential is not corrected.
**AP waveform** = median over valid spikes that come ≥ 0.5 s after the previous spike.

In [ ]:
features = feature_table(selection, spikes, bs, log)
display(features[["abf_filename", "cell_tag", "n_baseline_recordings_for_cell", "firing_class", "n_spikes",
                  "firing_rate_Hz", "isi_cv", "vm_baseline_mV", "ap_threshold_mV", "ap_half_width_ms", "flags"]])

In [ ]:
# Review: recording-level flags and flagged spikes
display(features.loc[features["flags"].fillna("") != "", ["abf_filename", "flags", "abf_tags"]])
if len(spikes):
    display(spikes[~spikes["valid"].astype(bool)])

## Step 4 · Save
`spikes.csv` (one row per spike), `detection.csv` (one row per recording: counts, flags, figure) and
`baseline_features.csv` (one row per recording). Several baseline recordings of the same cell are **not
averaged**: see `n_baseline_recordings_for_cell`.

In [ ]:
for df, p in zip((spikes, detection, features), targets):
    write_csv(stamp(df, log.run_id), p)
    print("written:", p)
log.finish(targets)